# Predict, Ship, Compete
## From SQL to Live A/B Test

**Your mission**: Build a model that decides which ads to show to which users to **maximize revenue** (not just clicks).

Your model will be deployed in a live A/B test against other teams. The team that generates the most revenue per impression wins.

---

> **Setup requirement: Python 3.12.** Models pickled under a different Python minor version (3.11, 3.13, ...) cannot be loaded by the server and will fail at upload with cryptic errors. If you're not on 3.12, create a fresh environment first: `conda create -n mayday python=3.12` (or `python3.12 -m venv .venv`). Then `pip install -r requirements.txt`.

**Server URL is set in the next cell** — pick a unique team name.

In [ ]:
import sys
from pathlib import Path

# Project root (works from repo root or notebooks/)
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_SOURCE = "local"  # "local" (offline) or "live" (workshop server)
SERVER = "https://lse-mayday.onrender.com"
TEAM_NAME = "The_Meat_Team"  # Pick a unique team name

if sys.version_info[:2] != (3, 12):
    print(f'WARNING: workshop server expects Python 3.12 — you are on '
          f'{sys.version_info.major}.{sys.version_info.minor}. Model upload '
          f'may fail with pickle errors.')


In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cloudpickle
import time
import io
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
from src.notebook_query import query, ensure_data, get_schema, print_schema

ensure_data()  # generates synthetic parquet tables on first run


### Register your team

In [ ]:
if DATA_SOURCE == "live":
    import requests
    resp = requests.post(
        f"{SERVER}/api/teams/{TEAM_NAME}/register",
        json={"members": ["AJ", "Anna", "Lynlee", "Eren"]},
    )
    print(resp.json())
else:
    print(f"Offline mode — skipping team registration.")
    print(f"Team name for model export: {TEAM_NAME}")


---

# Phase 1: Explore the Data (45 min)

You have access to an e-commerce ad database with four tables:
- **users** — 10,000 users with demographics and behavior
- **ads** — 200 ad creatives with product and creative metadata
- **impressions** — 500,000 ad impressions (did the user click?)
- **conversions** — purchases that followed a click

> **Offline mode:** SQL queries run locally against synthetic parquet data via DuckDB.
> Set `DATA_SOURCE = "live"` in cell 1 if the workshop server is available.

## 1.1 Get the lay of the land

In [ ]:
# Check what we're working with
print_schema(get_schema())

In [ ]:
# Overall funnel metrics
query("""
SELECT
  COUNT(*) as impressions,
  SUM(clicked) as clicks,
  ROUND(AVG(clicked) * 100, 2) as ctr_pct,
  (SELECT COUNT(*) FROM conversions) as conversions,
  (SELECT ROUND(SUM(revenue), 2) FROM conversions) as total_revenue
FROM impressions
""")

## 1.2 Investigate: Are all clicks equally valuable?

**Key question**: If you optimize purely for CTR (click-through rate), will you also maximize revenue?

In [ ]:
# CTR and revenue by user segments
# Try different groupings: loyalty_tier, device_type, age_group, etc.

segment_stats = query("""
SELECT
  u.loyalty_tier,
  u.device_type,
  COUNT(*) as impressions,
  SUM(i.clicked) as clicks,
  ROUND(AVG(i.clicked) * 100, 2) as ctr_pct,
  COUNT(c.conversion_id) as conversions,
  ROUND(SUM(c.revenue), 2) as total_revenue,
  ROUND(SUM(c.revenue) / COUNT(*), 4) as revenue_per_impression
FROM impressions i
JOIN users u ON i.user_id = u.user_id
LEFT JOIN conversions c ON i.impression_id = c.impression_id
GROUP BY u.loyalty_tier, u.device_type
ORDER BY revenue_per_impression DESC
""")
segment_stats

In [ ]:
# YOUR EXPLORATION: What patterns do you see?
# - Which segments have high CTR but low revenue?
# - Which segments have low CTR but high revenue per impression?
# - What does this mean for your modeling strategy?
#
# Write your own queries below:

my_stats = query("""
  WITH base AS (
  SELECT i.impression_id, i.clicked, u.loyalty_tier, u.device_type, u.age_group,
         CASE
           WHEN u.sessions_per_week < 2 THEN 'low'
           WHEN u.sessions_per_week < 5 THEN 'medium'
           ELSE 'high'
         END AS browsing_freq
  FROM impressions i
  JOIN users u ON i.user_id = u.user_id
), funnel AS (
  SELECT b.*, c.conversion_id, c.revenue
  FROM base b
  LEFT JOIN conversions c ON b.impression_id = c.impression_id
)
SELECT 'loyalty_tier' AS dimension, loyalty_tier AS segment,
       COUNT(*) AS impressions,
       SUM(clicked) AS clicks,
       ROUND(100.0*AVG(clicked),2) AS ctr_pct,
       COUNT(conversion_id) AS purchases,
       ROUND(100.0*COUNT(conversion_id)/NULLIF(SUM(clicked),0),2) AS click_to_purchase_pct,
       ROUND(SUM(COALESCE(revenue,0.0)),2) AS total_revenue,
       ROUND(SUM(COALESCE(revenue,0.0))/COUNT(*),4) AS revenue_per_impression
FROM funnel GROUP BY loyalty_tier
UNION ALL
SELECT 'device_type', device_type,
       COUNT(*), SUM(clicked), ROUND(100.0*AVG(clicked),2), COUNT(conversion_id),
       ROUND(100.0*COUNT(conversion_id)/NULLIF(SUM(clicked),0),2),
       ROUND(SUM(COALESCE(revenue,0.0)),2), ROUND(SUM(COALESCE(revenue,0.0))/COUNT(*),4)
FROM funnel GROUP BY device_type
UNION ALL
SELECT 'age_group', age_group,
       COUNT(*), SUM(clicked), ROUND(100.0*AVG(clicked),2), COUNT(conversion_id),
       ROUND(100.0*COUNT(conversion_id)/NULLIF(SUM(clicked),0),2),
       ROUND(SUM(COALESCE(revenue,0.0)),2), ROUND(SUM(COALESCE(revenue,0.0))/COUNT(*),4)
FROM funnel GROUP BY age_group
UNION ALL
SELECT 'browsing_freq', browsing_freq,
       COUNT(*), SUM(clicked), ROUND(100.0*AVG(clicked),2), COUNT(conversion_id),
       ROUND(100.0*COUNT(conversion_id)/NULLIF(SUM(clicked),0),2),
       ROUND(SUM(COALESCE(revenue,0.0)),2), ROUND(SUM(COALESCE(revenue,0.0))/COUNT(*),4)
FROM funnel GROUP BY browsing_freq
ORDER BY dimension, revenue_per_impression DESC;
""")

my_stats


In [ ]:
global_funnel = query("""
SELECT
  COUNT(*) AS impressions,
  SUM(i.clicked) AS clicks,
  ROUND(100.0 * AVG(i.clicked), 2) AS ctr_pct,
  COUNT(c.conversion_id) AS purchases,
  ROUND(100.0 * COUNT(c.conversion_id) / NULLIF(SUM(i.clicked), 0), 2) AS click_to_purchase_pct,
  ROUND(COALESCE(SUM(c.revenue), 0), 2) AS total_revenue,
  ROUND(COALESCE(SUM(c.revenue), 0) / COUNT(*), 4) AS revenue_per_impression,
  ROUND(COALESCE(SUM(c.revenue), 0) / NULLIF(COUNT(c.conversion_id), 0), 2) AS revenue_per_purchase
FROM impressions i
LEFT JOIN conversions c ON i.impression_id = c.impression_id
""")
global_funnel

In [ ]:
# 2) CTR vs monetization divergence by user dimensions
user_diagnostics = query("""
WITH base AS (
  SELECT
    i.impression_id, i.clicked,
    u.loyalty_tier, u.device_type, u.age_group,
    CASE
      WHEN u.sessions_per_week < 2 THEN 'low'
      WHEN u.sessions_per_week < 5 THEN 'medium'
      ELSE 'high'
    END AS browsing_freq
  FROM impressions i
  JOIN users u ON i.user_id = u.user_id
),
funnel AS (
  SELECT b.*, c.conversion_id, COALESCE(c.revenue, 0) AS revenue
  FROM base b
  LEFT JOIN conversions c ON b.impression_id = c.impression_id
)
SELECT 'loyalty_tier' AS dimension, loyalty_tier AS segment,
       COUNT(*) AS impressions, SUM(clicked) AS clicks,
       ROUND(100.0 * AVG(clicked), 2) AS ctr_pct,
       COUNT(conversion_id) AS purchases,
       ROUND(100.0 * COUNT(conversion_id) / NULLIF(SUM(clicked), 0), 2) AS click_to_purchase_pct,
       ROUND(SUM(revenue), 2) AS total_revenue,
       ROUND(SUM(revenue) / COUNT(*), 4) AS revenue_per_impression,
       ROUND(SUM(revenue) / NULLIF(SUM(clicked), 0), 4) AS revenue_per_click
FROM funnel GROUP BY loyalty_tier
UNION ALL
SELECT 'device_type', device_type,
       COUNT(*), SUM(clicked), ROUND(100.0 * AVG(clicked), 2), COUNT(conversion_id),
       ROUND(100.0 * COUNT(conversion_id) / NULLIF(SUM(clicked), 0), 2),
       ROUND(SUM(revenue), 2), ROUND(SUM(revenue) / COUNT(*), 4),
       ROUND(SUM(revenue) / NULLIF(SUM(clicked), 0), 4)
FROM funnel GROUP BY device_type
UNION ALL
SELECT 'age_group', age_group,
       COUNT(*), SUM(clicked), ROUND(100.0 * AVG(clicked), 2), COUNT(conversion_id),
       ROUND(100.0 * COUNT(conversion_id) / NULLIF(SUM(clicked), 0), 2),
       ROUND(SUM(revenue), 2), ROUND(SUM(revenue) / COUNT(*), 4),
       ROUND(SUM(revenue) / NULLIF(SUM(clicked), 0), 4)
FROM funnel GROUP BY age_group
UNION ALL
SELECT 'browsing_freq', browsing_freq,
       COUNT(*), SUM(clicked), ROUND(100.0 * AVG(clicked), 2), COUNT(conversion_id),
       ROUND(100.0 * COUNT(conversion_id) / NULLIF(SUM(clicked), 0), 2),
       ROUND(SUM(revenue), 2), ROUND(SUM(revenue) / COUNT(*), 4),
       ROUND(SUM(revenue) / NULLIF(SUM(clicked), 0), 4)
FROM funnel GROUP BY browsing_freq
ORDER BY dimension, revenue_per_impression DESC
""")
user_diagnostics

In [ ]:
# 3) Ad quality/clickbait matrix (0-10 scale buckets)
creative_matrix = query("""
WITH x AS (
  SELECT
    CASE
      WHEN a.creative_quality_score < 2 THEN 'Q:0-2'
      WHEN a.creative_quality_score < 4 THEN 'Q:2-4'
      WHEN a.creative_quality_score < 6 THEN 'Q:4-6'
      WHEN a.creative_quality_score < 8 THEN 'Q:6-8'
      ELSE 'Q:8-10'
    END AS quality_bucket,
    CASE
      WHEN a.headline_clickbait_score < 2 THEN 'C:0-2'
      WHEN a.headline_clickbait_score < 4 THEN 'C:2-4'
      WHEN a.headline_clickbait_score < 6 THEN 'C:4-6'
      WHEN a.headline_clickbait_score < 8 THEN 'C:6-8'
      ELSE 'C:8-10'
    END AS clickbait_bucket,
    i.clicked,
    COALESCE(c.revenue, 0) AS revenue
  FROM impressions i
  JOIN ads a ON i.ad_id = a.ad_id
  LEFT JOIN conversions c ON i.impression_id = c.impression_id
)
SELECT
  quality_bucket,
  clickbait_bucket,
  COUNT(*) AS impressions,
  ROUND(100.0 * AVG(clicked), 2) AS ctr_pct,
  ROUND(SUM(revenue) / COUNT(*), 4) AS revenue_per_impression,
  ROUND(SUM(revenue) / NULLIF(SUM(clicked), 0), 4) AS revenue_per_click
FROM x
GROUP BY quality_bucket, clickbait_bucket
HAVING COUNT(*) >= 100
ORDER BY quality_bucket, clickbait_bucket
""")
creative_matrix

In [ ]:
# 4) Context effects (placement + time)
context_diagnostics = query("""
SELECT
  i.page_type,
  i.position,
  i.day_of_week,
  i.hour_of_day,
  COUNT(*) AS impressions,
  ROUND(100.0 * AVG(i.clicked), 2) AS ctr_pct,
  ROUND(COALESCE(SUM(c.revenue), 0) / COUNT(*), 4) AS revenue_per_impression
FROM impressions i
LEFT JOIN conversions c ON i.impression_id = c.impression_id
GROUP BY i.page_type, i.position, i.day_of_week, i.hour_of_day
HAVING COUNT(*) >= 150
ORDER BY revenue_per_impression DESC
LIMIT 150
""")
context_diagnostics

In [ ]:
# 5) Candidate ad-level targets (enough volume only)
ad_candidates = query("""
SELECT
  i.ad_id,
  a.category,
  a.ad_format,
  ROUND(a.creative_quality_score, 2) AS creative_quality_score,
  ROUND(a.headline_clickbait_score, 2) AS headline_clickbait_score,
  COUNT(*) AS impressions,
  ROUND(100.0 * AVG(i.clicked), 2) AS ctr_pct,
  ROUND(COALESCE(SUM(c.revenue), 0), 2) AS total_revenue,
  ROUND(COALESCE(SUM(c.revenue), 0) / COUNT(*), 4) AS revenue_per_impression,
  ROUND(COALESCE(SUM(c.revenue), 0) / NULLIF(SUM(i.clicked), 0), 4) AS revenue_per_click
FROM impressions i
JOIN ads a ON i.ad_id = a.ad_id
LEFT JOIN conversions c ON i.impression_id = c.impression_id
GROUP BY i.ad_id, a.category, a.ad_format, a.creative_quality_score, a.headline_clickbait_score
HAVING COUNT(*) >= 200
ORDER BY revenue_per_impression DESC
LIMIT 50
""")
ad_candidates

In [ ]:
# Hint: do clickbait-y ads generate more revenue?
# Try grouping by clickbait score buckets and comparing CTR with
# revenue per impression. The pattern might surprise you.

clickbait_analysis = query("""
WITH buckets AS (
  SELECT 1 AS bucket_id, '0-2 (low)' AS clickbait_bucket, 0.0 AS lo, 2.0 AS hi UNION ALL
  SELECT 2, '2-4', 2.0, 4.0 UNION ALL
  SELECT 3, '4-6', 4.0, 6.0 UNION ALL
  SELECT 4, '6-8', 6.0, 8.0 UNION ALL
  SELECT 5, '8-10 (high)', 8.0, 10.000001
),
agg AS (
  SELECT
    CASE
      WHEN a.headline_clickbait_score < 2 THEN 1
      WHEN a.headline_clickbait_score < 4 THEN 2
      WHEN a.headline_clickbait_score < 6 THEN 3
      WHEN a.headline_clickbait_score < 8 THEN 4
      ELSE 5
    END AS bucket_id,
    COUNT(*) AS impressions,
    SUM(i.clicked) AS clicks,
    AVG(i.clicked) * 100.0 AS ctr_pct,
    COUNT(c.conversion_id) AS conversions,
    COALESCE(SUM(c.revenue), 0) AS total_revenue
  FROM impressions i
  JOIN ads a ON i.ad_id = a.ad_id
  LEFT JOIN conversions c ON i.impression_id = c.impression_id
  GROUP BY 1
)
SELECT
  b.clickbait_bucket,
  COALESCE(a.impressions, 0) AS impressions,
  COALESCE(a.clicks, 0) AS clicks,
  ROUND(COALESCE(a.ctr_pct, 0), 2) AS ctr_pct,
  COALESCE(a.conversions, 0) AS conversions,
  ROUND(COALESCE(a.total_revenue, 0), 2) AS total_revenue,
  ROUND(COALESCE(a.total_revenue, 0) / NULLIF(COALESCE(a.impressions, 0), 0), 4) AS revenue_per_impression
FROM buckets b
LEFT JOIN agg a ON b.bucket_id = a.bucket_id
ORDER BY b.bucket_id
""")
clickbait_analysis

### Insight check

Before moving on, make sure you can answer:
1. Do high-CTR user segments also have the highest revenue per impression?
2. Do high-clickbait ads generate more revenue than high-quality ads?
3. What is the conversion rate (given click) for different user types?

**The answer to these questions should inform what your model optimizes for.**

---

# Phase 2: Build Your Model (75 min)

## 2.1 Pull training data

Pull the full dataset with user, ad, and context features joined together.

In [ ]:
from src.data import pull_training_data

# Load joined training data from local parquet (or live API if DATA_SOURCE == "live")
print("Loading training data...")
df = pull_training_data(max_rows=100_000, source=DATA_SOURCE)
print(f"\nTotal: {len(df):,} rows")
df.head()

In [ ]:
# Worth checking before you train anything:
#   - What's the overall CTR? CVR given click?
#   - What's average revenue per impression? Per conversion?
#   - Does df.head() / df.describe() look sensible?
#   - Any nulls or weird values?

## 2.2 Feature engineering

Prepare features for your model. The model will need to accept a DataFrame with these columns during inference.

In [ ]:
def prepare_features(df):
    """Prepare features for modelling.

    The same function will be used at inference time — it must work on
    a single row or a batch and produce the same column order both times.
    """
    # e.g. make categorical variables (age_group, region, category, ...)
    # into dummies; pass numeric ones through as-is. Return a DataFrame
    # ready for your model.
    raise NotImplementedError

X = prepare_features(df)
print(f"Feature matrix: {X.shape}")

## 2.3 Choose your target

This is the most important decision. What should your model predict?

The A/B test scores you on **revenue per impression** — but `revenue`
itself is sparse and noisy, so regressing on it directly isn't always
the best move. Think carefully about what target most closely tracks
the metric you'll actually be judged on.

In [ ]:
# Pick a target (or build one):
y_click = df['clicked'].values
y_revenue = df['revenue'].values

# Hint: revenue per impression is a chain of conditional events.
# What has to happen first for revenue to exist at all? Then what?
# How would you model each step?

In [ ]:
# Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_click, test_size=0.2, random_state=42  # Change target as needed
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2.4 Train your model

Start simple and iterate. Remember: your model needs to be fast at inference time (Phase 3).

In [ ]:
# Example: Logistic Regression (fast, interpretable baseline)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict_proba(X_test)[:, 1]
print(f"Logistic Regression:")
print(f"  AUC: {roc_auc_score(y_test, y_pred_lr):.4f}")
print(f"  Log Loss: {log_loss(y_test, y_pred_lr):.4f}")

In [ ]:
# Other classifiers / regressors (RandomForest, LightGBM, XGBoost,
# neural nets) are all fine — the server accepts anything with a
# .predict() method. LightGBM is a strong default if you want more
# capacity than logistic regression.

In [ ]:
# YOUR TURN: Try different models, features, targets, hyperparameters
# Ideas:
#   - Random Forest, XGBoost, neural network
#   - Add interaction features (e.g., user_aov * product_price)
#   - Train separate CTR and CVR models, combine them
#   - Use regression on revenue instead of classification
#   - Weight samples by revenue potential



## 2.5 Evaluate on the right metric

AUC measures ranking quality. But we care about revenue. Let's evaluate properly.

In [ ]:
# AUC measures ranking on click — but you're scored on revenue per
# impression. A better local check: take the top N% of your test set
# by your model's score, and compare their mean revenue to the rest.
# A useful model should put high-revenue impressions near the top.

---

# Phase 3: Optimize for Production (30 min)

Your model must score ads in real-time. The simulation has a **latency budget** (default: 50ms).
If your model is slower, some of your traffic gets a random ad choice instead — penalizing slow models.

## 3.1 Benchmark your model's latency

In [ ]:
def benchmark_latency(model, X_sample, n_trials=100):
    """Measure inference latency for a batch of 10 rows (simulates one request)."""
    batch = X_sample.head(10)  # The simulation scores 10 candidate ads per request
    latencies = []

    for _ in range(n_trials):
        start = time.perf_counter()
        if hasattr(model, 'predict_proba'):
            model.predict_proba(batch)
        else:
            model.predict(batch)
        elapsed_ms = (time.perf_counter() - start) * 1000
        latencies.append(elapsed_ms)

    latencies = np.array(latencies)
    print(f"Latency (batch of 10):")
    print(f"  Median: {np.median(latencies):.2f} ms")
    print(f"  P95:    {np.percentile(latencies, 95):.2f} ms")
    print(f"  P99:    {np.percentile(latencies, 99):.2f} ms")
    print(f"  Max:    {np.max(latencies):.2f} ms")

    budget = 50  # ms
    violations = (latencies > budget).mean()
    print(f"\n  Budget violations (>{budget}ms): {violations:.1%}")
    if violations > 0.1:
        print(f"  WARNING: >10% of requests will be penalized!")
    return latencies

print("Logistic Regression:")
benchmark_latency(lr, X_test)
print("\nLightGBM:")
benchmark_latency(lgb_model, X_test)

In [ ]:
# If your model is too slow, try:
#   - Fewer trees / smaller ensemble
#   - Fewer features (drop low-importance ones)
#   - Simpler model (logistic regression is very fast)
#   - Quantize or compress
#
# Feature importance (for feature selection):
if hasattr(lgb_model, 'feature_importances_'):
    importance = pd.Series(
        lgb_model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=True)
    importance.tail(20).plot.barh(figsize=(10, 6), color='#6c63ff')
    plt.title('Top 20 Feature Importances')
    plt.tight_layout()
    plt.show()

## 3.2 Wrap your model for deployment

The simulator calls `model.predict(features_df)` where `features_df` has the same columns as your training data.
The prediction should be a **score** — higher = better ad to show. This could be:
- P(click) from a CTR model
- P(click) * P(convert|click) from two models combined
- Predicted revenue
- Any scoring function you design

In [ ]:
class ScoringModel:
    """Wraps your trained model(s) into a scoring function for the simulator."""

    def __init__(self, model):
        # You can store multiple models here if you want a composite
        # score (e.g. self.ctr_model, self.cvr_model).
        self.model = model

    def predict(self, X):
        # Return a 1D array of scores — higher = better ad to show.
        # If `model` is a classifier, use predict_proba(X)[:, 1].
        return self.model.predict_proba(X)[:, 1]


final_model = ScoringModel(lr)  # or whatever you trained

# Verify it returns sensible scores
test_scores = final_model.predict(X_test.head(10))
print(f"Sample scores: {test_scores}")

In [ ]:
# Final latency check
print("Final model latency:")
benchmark_latency(final_model, X_test)

---

# Phase 4: Deploy & Compete! (45 min)

## 4.1 Save and upload your model

In [ ]:
from pathlib import Path

# Save model using cloudpickle (captures class definitions for the server)
model_path = f"model_{TEAM_NAME}.pkl"
with open(model_path, 'wb') as f:
    cloudpickle.dump(final_model, f)

print(f"Model saved to {model_path}")
print(f"Size: {Path(model_path).stat().st_size / 1024:.1f} KB")

In [ ]:
from pathlib import Path

# Upload to the server (live mode only)
if DATA_SOURCE == "live":
    with open(model_path, 'rb') as f:
        resp = requests.post(
            f"{SERVER}/api/teams/{TEAM_NAME}/model",
            files={"model": (model_path, f, "application/octet-stream")}
        )

    result = resp.json()
    print(f"Upload result: {result}")
    if resp.ok:
        print(f"\nModel uploaded successfully!")
        print(f"  Type: {result.get('model_type')}")
        print(f"  Validation prediction: {result.get('validation_prediction'):.4f}")
        print(f"  Validation latency: {result.get('validation_latency_ms'):.2f} ms")
    else:
        print(f"\nERROR: {result}")
else:
    print("Offline mode — model saved locally. Run `streamlit run app/streamlit_app.py` to demo scoring.")


In [ ]:
# # mayday_model.py
# # ───────────────
# # Full pipeline: pull data → engineer features → train 3-stage chain
# # → evaluate → benchmark latency → upload to server.

# # Usage:
# #     python mayday_model.py --team your-team-name
# #     python mayday_model.py --team your-team-name --skip-upload   # dry run
# #     python mayday_model.py --team your-team-name --rows 200000   # more data
# # """

# import sys
# if sys.version_info[:2] != (3, 12):
#     print(
#         f"WARNING: server requires Python 3.12 — you are on "
#         f"{sys.version_info.major}.{sys.version_info.minor}. "
#         f"Pickle may fail on upload."
#     )

# import argparse
# import time
# import warnings
# import numpy as np
# import pandas as pd
# import requests
# import cloudpickle
# from pathlib import Path
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import roc_auc_score
# from sklearn.preprocessing import LabelEncoder
# from xgboost import XGBClassifier, XGBRegressor

# warnings.filterwarnings("ignore")

# # ── Config ───────────────────────────────────────────────────────────────────

# SERVER    = "https://lse-mayday.onrender.com"
# TEAM_NAME = "The_Meat_Team" 

# CAT_COLS  = [
#     "age_group", "gender", "device_type", "region",
#     "loyalty_tier", "category", "ad_format", "page_type",
# ]
# ENCODERS: dict = {}


# # ── 1. Data pulling ───────────────────────────────────────────────────────────

# def query(sql: str, limit: int = 50_000) -> pd.DataFrame:
#     resp = requests.post(f"{SERVER}/api/sql", json={"query": sql, "limit": limit})
#     resp.raise_for_status()
#     data = resp.json()
#     return pd.DataFrame(data["rows"], columns=data["columns"])


# def pull_training_data(max_rows: int = 100_000) -> pd.DataFrame:
#     batch_size = 50_000
#     chunks = []
#     offset = 0
#     while offset < max_rows:
#         batch = min(batch_size, max_rows - offset)
#         df = query(f"""
#             SELECT
#                 i.impression_id,
#                 u.age_group, u.gender, u.device_type, u.region,
#                 u.account_age_days, u.past_purchases, u.avg_order_value,
#                 u.sessions_per_week, u.loyalty_tier,
#                 a.category, a.ad_format, a.product_price, a.discount_pct,
#                 a.creative_quality_score, a.headline_clickbait_score,
#                 a.brand_familiarity,
#                 i.page_type, i.position, i.hour_of_day, i.day_of_week,
#                 i.session_depth,
#                 i.clicked,
#                 CASE WHEN c.conversion_id IS NOT NULL THEN 1 ELSE 0 END AS converted,
#                 COALESCE(c.revenue, 0) AS revenue
#             FROM impressions i
#             JOIN users u ON i.user_id = u.user_id
#             JOIN ads   a ON i.ad_id   = a.ad_id
#             LEFT JOIN conversions c ON i.impression_id = c.impression_id
#             ORDER BY i.impression_id
#             LIMIT {batch} OFFSET {offset}
#         """, limit=batch)
#         if len(df) == 0:
#             break
#         chunks.append(df)
#         offset += len(df)
#         print(f"  pulled {offset:,} rows...")
#     return pd.concat(chunks, ignore_index=True)


# # ── 2. Feature engineering ────────────────────────────────────────────────────

# FEATURE_COLS = [
#     # user
#     "age_group", "gender", "device_type", "region",
#     "loyalty_tier", "loyalty_rank",
#     "account_age_days", "past_purchases", "avg_order_value", "sessions_per_week",
#     # ad
#     "category", "ad_format", "product_price", "discount_pct",
#     "max_possible_revenue", "price_x_quality", "discount_x_brand",
#     "creative_quality_score", "headline_clickbait_score", "brand_familiarity",
#     # context
#     "page_type", "position", "hour_of_day", "day_of_week",
#     "session_depth", "is_weekend", "is_peak_hour",
#     # interactions
#     "aov_x_price", "purchases_x_brand",
# ]


# def prepare_features(df: pd.DataFrame, fit: bool = False) -> pd.DataFrame:
#     """
#     Transform raw joined DataFrame into model-ready feature matrix.
#     Call with fit=True once on training data; False at inference time.
#     """
#     d = df.copy()

#     # loyalty ordinal
#     loyalty_map = {"bronze": 0, "silver": 1, "gold": 2, "platinum": 3}
#     d["loyalty_rank"] = d["loyalty_tier"].str.lower().map(loyalty_map).fillna(0)

#     # ad revenue ceiling
#     d["max_possible_revenue"] = d["product_price"] * (1 - d["discount_pct"] / 100)
#     d["price_x_quality"]      = d["product_price"] * d["creative_quality_score"]
#     d["discount_x_brand"]     = d["discount_pct"]  * d["brand_familiarity"]

#     # user × ad interactions
#     d["aov_x_price"]       = d["avg_order_value"] * d["product_price"]
#     d["purchases_x_brand"] = d["past_purchases"]  * d["brand_familiarity"]

#     # time context
#     d["is_weekend"]   = d["day_of_week"].isin([5, 6]).astype(int)
#     d["is_peak_hour"] = d["hour_of_day"].between(18, 22).astype(int)

#     # encode categoricals
#     for col in CAT_COLS:
#         d[col] = d[col].astype(str)
#         if fit:
#             le = LabelEncoder()
#             d[col] = le.fit_transform(d[col])
#             ENCODERS[col] = le
#         else:
#             le = ENCODERS[col]
#             d[col] = d[col].map(
#                 lambda x, le=le: _le.transform([x])[0] if x in _le.classes else -1
#             )

#     return d[FEATURE_COLS].astype(float)


# # ── 3. Three-stage model ──────────────────────────────────────────────────────

# def train_stages(df_train: pd.DataFrame, X_train: pd.DataFrame):
#     """Train P(click), P(convert|click), E(revenue|convert)."""

#     xgb_shared = dict(
#         max_depth=5, learning_rate=0.05, n_estimators=300,
#         subsample=0.8, colsample_bytree=0.8,
#         tree_method="hist", random_state=42, n_jobs=-1,
#     )

#     # Stage 1: P(click)
#     print("\nStage 1 — P(click)")
#     spw1 = (df_train["clicked"] == 0).sum() / (df_train["clicked"] == 1).sum()
#     m1   = XGBClassifier(**xgb_shared, scale_pos_weight=spw1)
#     m1.fit(X_train, df_train["clicked"], verbose=False)

#     # Stage 2: P(convert | click)  — train only on clicked rows
#     print("Stage 2 — P(convert | click)")
#     mask_click = df_train["clicked"] == 1
#     spw2 = (
#         (df_train.loc[mask_click, "converted"] == 0).sum() /
#         (df_train.loc[mask_click, "converted"] == 1).sum()
#     )
#     m2 = XGBClassifier(**xgb_shared, scale_pos_weight=spw2)
#     m2.fit(X_train[mask_click], df_train.loc[mask_click, "converted"], verbose=False)

#     # Stage 3: E(revenue | convert)  — train only on converted rows
#     print("Stage 3 — E(revenue | convert)")
#     mask_conv = df_train["converted"] == 1
#     m3 = XGBRegressor(**xgb_shared)
#     m3.fit(X_train[mask_conv], df_train.loc[mask_conv, "revenue"])

#     return m1, m2, m3


# def evaluate(m1, m2, m3, df_test: pd.DataFrame, X_test: pd.DataFrame):
#     """Print AUC per stage and the bucket revenue lift test."""
#     click_auc = roc_auc_score(df_test["clicked"], m1.predict_proba(X_test)[:, 1])

#     mask_click = df_test["clicked"] == 1
#     conv_auc   = roc_auc_score(
#         df_test.loc[mask_click, "converted"],
#         m2.predict_proba(X_test[mask_click])[:, 1],
#     )

#     p_click   = m1.predict_proba(X_test)[:, 1]
#     p_convert = m2.predict_proba(X_test)[:, 1]
#     e_revenue = m3.predict(X_test)
#     score     = p_click * p_convert * e_revenue

#     df_eval          = df_test.copy()
#     df_eval["score"] = score
#     threshold        = df_eval["score"].quantile(0.9)

#     top_rpi = df_eval.loc[df_eval["score"] >= threshold, "revenue"].mean()
#     bot_rpi = df_eval.loc[df_eval["score"] <  threshold, "revenue"].mean()

#     print(f"\n{'─'*40}")
#     print(f"  Stage 1 AUC (click):          {click_auc:.4f}")
#     print(f"  Stage 2 AUC (convert|click):  {conv_auc:.4f}")
#     print(f"  Top 10% avg revenue:          £{top_rpi:.4f}")
#     print(f"  Bottom 90% avg revenue:       £{bot_rpi:.4f}")
#     print(f"  Revenue lift:                 {top_rpi/bot_rpi:.2f}×")
#     print(f"{'─'*40}")

#     if top_rpi / bot_rpi < 2:
#         print("  ⚠  Lift <2× — check target or feature engineering")
#     elif top_rpi / bot_rpi > 3:
#         print("  ✓  Lift >3× — good signal")

#     return score


# # ── 4. Scoring wrapper ────────────────────────────────────────────────────────

# class ScoringModel:
#     """
#     Deployed wrapper. Server calls model.predict(raw_df) and expects
#     a 1-D array of scores — higher = better ad to show.
#     """
#     def _init_(self, m1, m2, m3, encoders: dict, feature_cols: list):
#         self.m1           = m1
#         self.m2           = m2
#         self.m3           = m3
#         self.encoders     = encoders
#         self.feature_cols = feature_cols

#     def _prepare(self, df: pd.DataFrame) -> pd.DataFrame:
#         d = df.copy()
#         loyalty_map = {"bronze": 0, "silver": 1, "gold": 2, "platinum": 3}
#         d["loyalty_rank"]        = d["loyalty_tier"].str.lower().map(loyalty_map).fillna(0)
#         d["max_possible_revenue"] = d["product_price"] * (1 - d["discount_pct"] / 100)
#         d["price_x_quality"]      = d["product_price"] * d["creative_quality_score"]
#         d["discount_x_brand"]     = d["discount_pct"]  * d["brand_familiarity"]
#         d["aov_x_price"]          = d["avg_order_value"] * d["product_price"]
#         d["purchases_x_brand"]    = d["past_purchases"]  * d["brand_familiarity"]
#         d["is_weekend"]           = d["day_of_week"].isin([5, 6]).astype(int)
#         d["is_peak_hour"]         = d["hour_of_day"].between(18, 22).astype(int)
#         for col in CAT_COLS:
#             le   = self.encoders[col]
#             d[col] = d[col].astype(str).map(
#                 lambda x, le=le: _le.transform([x])[0] if x in _le.classes else -1
#             )
#         return d[self.feature_cols].astype(float)

#     def predict(self, raw_df: pd.DataFrame) -> np.ndarray:
#         X         = self._prepare(raw_df)
#         p_click   = self.m1.predict_proba(X)[:, 1]
#         p_convert = self.m2.predict_proba(X)[:, 1]
#         e_revenue = self.m3.predict(X)
#         return p_click * p_convert * e_revenue


# # ── 5. Latency benchmark ──────────────────────────────────────────────────────

# def benchmark_latency(model: ScoringModel, sample_df: pd.DataFrame, n_trials: int = 100):
#     batch      = sample_df.head(10)
#     latencies  = []
#     for _ in range(n_trials):
#         t0 = time.perf_counter()
#         model.predict(batch)
#         latencies.append((time.perf_counter() - t0) * 1000)

#     latencies  = np.array(latencies)
#     violations = (latencies > 50).mean()
#     print(f"\nLatency over {n_trials} trials (batch=10):")
#     print(f"  Median: {np.median(latencies):.2f} ms")
#     print(f"  P95:    {np.percentile(latencies, 95):.2f} ms")
#     print(f"  P99:    {np.percentile(latencies, 99):.2f} ms")
#     print(f"  Budget violations (>50ms): {violations:.1%}")
#     if violations > 0.1:
#         print("  ⚠  >10% violations — reduce n_estimators or drop features")
#     else:
#         print("  ✓  Within latency budget")


# # ── 6. Upload ─────────────────────────────────────────────────────────────────

# def upload_model(model: ScoringModel, team: str):
#     path = f"model_{team}.pkl"
#     with open(path, "wb") as f:
#         cloudpickle.dump(model, f)
#     size_kb = Path(path).stat().st_size / 1024
#     print(f"\nModel saved: {path} ({size_kb:.1f} KB)")

#     with open(path, "rb") as f:
#         resp = requests.post(
#             f"{SERVER}/api/teams/{team}/model",
#             files={"model": (path, f, "application/octet-stream")},
#         )
#     result = resp.json()
#     if resp.ok:
#         print(f"✓ Uploaded successfully")
#         print(f"  Model type:           {result.get('model_type')}")
#         print(f"  Validation score:     {result.get('validation_prediction'):.4f}")
#         print(f"  Validation latency:   {result.get('validation_latency_ms'):.2f} ms")
#     else:
#         print(f"✗ Upload failed: {result}")


# # ── Main ──────────────────────────────────────────────────────────────────────

# def main():
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--team",        default=TEAM_NAME)
#     parser.add_argument("--rows",        type=int, default=100_000)
#     parser.add_argument("--skip-upload", action="store_true")
#     args = parser.parse_args()

#     print(f"Team: {args.team} | Rows: {args.rows:,} | Upload: {not args.skip_upload}")

#     # Pull
#     print("\n── Pulling data ─────────────────────────────────────────────────")
#     df = pull_training_data(max_rows=args.rows)
#     print(f"Total rows: {len(df):,}")
#     print(f"CTR:        {df['clicked'].mean()*100:.2f}%")
#     print(f"CVR:        {df['converted'].mean()*100:.2f}%")
#     print(f"Avg rev/imp: £{df['revenue'].mean():.4f}")

#     # Features
#     print("\n── Engineering features ─────────────────────────────────────────")
#     X = prepare_features(df, fit=True)
#     print(f"Feature matrix: {X.shape}")

#     # Split
#     X_train, X_test, df_train, df_test = train_test_split(
#         X, df, test_size=0.2, random_state=42
#     )

#     # Train
#     print("\n── Training three-stage chain ───────────────────────────────────")
#     m1, m2, m3 = train_stages(df_train, X_train)

#     # Evaluate
#     print("\n── Evaluation ───────────────────────────────────────────────────")
#     evaluate(m1, m2, m3, df_test, X_test)

#     # Wrap
#     final_model = ScoringModel(m1, m2, m3, ENCODERS, FEATURE_COLS)

#     # Sanity check on raw df rows (simulates server calling predict on raw data)
#     sample_scores = final_model.predict(df.head(5))
#     print(f"\nSample scores (raw df): {np.round(sample_scores, 4)}")

#     # Latency
#     print("\n── Latency benchmark ────────────────────────────────────────────")
#     benchmark_latency(final_model, df.head(100))

#     # Upload
#     if not args.skip_upload:
#         print("\n── Uploading ─────────────────────────────────────────────────")
#         upload_model(final_model, args.team)
#     else:
#         print("\n── Skipping upload (--skip-upload) ──────────────────────────")


# if _name_ == "_main_":
#     main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, mean_absolute_error

# ── 1. Load data ─────────────────────────────────────────────────
df = pd.read_csv('impressions_merged.csv')

# ── 2. Feature engineering ───────────────────────────────────────
df['device_tier'] = df['device_type'] + '_' + df['loyalty_tier']
df['net_value'] = df['product_price'] * (1 - df['discount_pct'] / 100)
df['hour_bin'] = pd.cut(df['hour_of_day'], bins=[-1,6,12,18,23], labels=['night','morning','afternoon','evening'])
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

# ── 3. Preprocessing pipeline ────────────────────────────────────
num_features = ['past_purchases', 'session_depth', 'product_price', 'discount_pct', 'creative_quality', 'net_value']
cat_features = ['loyalty_tier', 'device_type', 'category', 'ad_format', 'hour_bin', 'is_weekend']
interaction_features = ['device_tier']

all_features = num_features + cat_features + interaction_features

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features + interaction_features),
])

# ── 4. Train/test split ──────────────────────────────────────────
X = df[all_features]
y_click = df['clicked']
y_convert = df['conversion_id'].notna().astype(int)
y_revenue = df['revenue'].fillna(0)

X_train, X_test, y_click_train, y_click_test,
    y_cv_train, y_cv_test, y_rev_train, y_rev_test = train_test_split(
    X, y_click, y_convert, y_revenue,
    test_size=0.2, random_state=42, stratify=y_convert)

# ── 5a. Click model ──────────────────────────────────────────────
clf_click = Pipeline([
    ('prep', preprocessor),
    ('clf', CalibratedClassifierCV(LogisticRegression(max_iter=1000, C=1.0), cv=5)),
])
clf_click.fit(X_train, y_click_train)
p_click = clf_click.predict_proba(X_test)[:, 1]

print(f'Click AUC: {roc_auc_score(y_click_test, p_click):.4f}')

# ── 5b. Conversion model (trained on clicks only) ────────────────
clicked_train = X_train[y_click_train == 1]
clicked_cv = y_cv_train[y_click_train == 1]

clf_convert = Pipeline([
    ('prep', preprocessor.set_output(transform='pandas')),
    ('clf', CalibratedClassifierCV(LogisticRegression(max_iter=1000, C=1.0), cv=5)),
])
clf_convert.fit(clicked_train, clicked_cv)

clicked_test = X_test[y_click_test == 1]
p_convert = clf_convert.predict_proba(clicked_test)[:, 1]
print(f'Convert AUC (on clicks): {roc_auc_score(y_cv_test[y_click_test == 1], p_convert):.4f}')

# ── 5c. Revenue regression (trained on conversions only) ─────────
converted_train = X_train[y_cv_train == 1]
revenue_train = y_rev_train[y_cv_train == 1]
revenue_train_log = np.log1p(revenue_train)  # log-transform to reduce skew

reg_revenue = Pipeline([
    ('prep', preprocessor),
    ('reg', lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31)),
])
reg_revenue.fit(converted_train, revenue_train_log)

# ── 6. Combined scoring function ─────────────────────────────────
def score_impressions(X):
    p_c = clf_click.predict_proba(X)[:, 1]
    p_v = clf_convert.predict_proba(X)[:, 1]
    e_rev = reg_revenue.predict(X)
    e_rev = np.expm1(e_rev)  # reverse log transform
    return p_c * p_v * e_rev  # expected revenue per impression

scores_test = score_impressions(X_test)

# ── 7. Bucket evaluation (the real metric) ───────────────────────
threshold = np.percentile(scores_test, 90)
top10 = y_rev_test[scores_test >= threshold]
bot90 = y_rev_test[scores_test < threshold]

print(f'Top 10% avg RPI:    £{top10.mean():.4f}')
print(f'Bottom 90% avg RPI: £{bot90.mean():.4f}')
print(f'Lift ratio:         {top10.mean() / bot90.mean():.2f}x')
# ── 8. Save models ───────────────────────────────────────────────
import joblib
joblib.dump({
    'click': clf_click,
    'convert': clf_convert,
    'revenue': reg_revenue,
}, 'ad_revenue_model.pkl')
print('Models saved to ad_revenue_model.pkl')

## 4.2 Watch the live dashboard

> **Offline mode:** The workshop server is unavailable. Use the Streamlit demo instead:
> `streamlit run app/streamlit_app.py`

When the server is live, open the dashboard to watch the A/B test in real time.

**Dashboard URL**: `{SERVER}/dashboard`

In [ ]:
if DATA_SOURCE == "live":
    resp = requests.get(f"{SERVER}/api/leaderboard")
    print(resp.json())
else:
    print("Offline mode — no live leaderboard. Compare models offline using revenue lift in the evaluation cells.")
